In [1]:
import os
os.environ.setdefault("USER_AGENT", "AI Agents and Agentic Workflows educational RAG notebook")

from langchain_community.document_loaders import WikipediaLoader, WebBaseLoader, Docx2txtLoader, PyPDFLoader, TextLoader, DirectoryLoader

from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import ChatOllama

## Setting up vector database and embeddings

In [2]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0)
embeddings_model = None  # Use Chroma's local all-MiniLM-L6-v2 embeddings
vector_db = Chroma("tourist_info", embeddings_model)

> **Chroma defaults used here (verified with the installed Chroma 1.3.0):** For a newly created local/single-node collection, `embeddings_model = None` lets Chroma attach its built-in `DefaultEmbeddingFunction`, which uses ONNX Runtime with `all-MiniLM-L6-v2`. The main dense-vector index is **HNSW** (approximate nearest-neighbor search), not IVF or PQ. Its default `space` is **`l2`**, which Chroma defines as squared Euclidean distance: $\sum_i (A_i-B_i)^2$. Therefore, smaller returned distances mean closer matches; this is not cosine similarity or dot product. Newly added vectors first enter a small brute-force buffer (default batch size: 100) before being merged into HNSW. These settings are established when the collection is created.

> Sources: [Chroma index configuration](https://docs.trychroma.com/docs/collections/configure) and [Chroma collection defaults](https://cookbook.chromadb.dev/core/collections/).

In [3]:
try:
    wikipedia_loader = WikipediaLoader(query="Paestum")
    wikipedia_chunks = text_splitter.split_documents(wikipedia_loader.load())
    vector_db.add_documents(wikipedia_chunks)
except Exception as error:
    print(f"Wikipedia API failed ({type(error).__name__}: {error}). Loading the Paestum page directly.")
    wikipedia_loader = WebBaseLoader(
        "https://en.wikipedia.org/wiki/Paestum",
        header_template={"User-Agent": "AI Agents and Agentic Workflows educational RAG notebook"}
    )
    wikipedia_chunks = text_splitter.split_documents(wikipedia_loader.load())
    vector_db.add_documents(wikipedia_chunks)

Wikipedia API failed (JSONDecodeError: Expecting value: line 1 column 1 (char 0)). Loading the Paestum page directly.


In [4]:
word_loader = Docx2txtLoader("Paestum/Paestum-Britannica.docx")
word_chunks = text_splitter.split_documents(word_loader.load())
vector_db.add_documents(word_chunks)

['2c5e4472-28bd-4602-a8c5-df610a1a902a',
 '818be703-fa15-41fa-81e8-752b2965016d',
 '2a2fd168-8ac4-494d-bf13-7863e18315ed',
 '1c337c6a-6752-44b3-8155-c82914982480',
 '37434785-ceb7-4888-84fb-6d6daac0b265',
 'b0c30a58-ba34-4e79-9cd1-11b785f9fd39',
 '9d9c3e24-0426-49a8-bc7c-66adc2dde510',
 'ff409946-4c1a-4382-b4ec-393c777c3186']

In [5]:
pdf_loader = PyPDFLoader("Paestum/PaestumRevisited.pdf")
pdf_chunks = text_splitter.split_documents(pdf_loader.load())
vector_db.add_documents(pdf_chunks)

['03c27343-3a21-4d2a-aa0c-379c5941d379',
 '7ea4d990-ca4f-4593-bbb0-a5ab882e2883',
 '84491ea3-3f9e-43d9-9bdf-dffc8df10e51',
 '3192c5dd-5e08-4077-bacb-565976cea00e',
 'd2fd6b92-4c23-4330-82f9-15dc6ff7e5d5',
 'b955dc81-93e5-42c1-be34-ffcae18d05a0',
 '7ce31370-1ea0-4008-ae51-fc14317b72d2',
 '7a8e32a7-c6b6-4747-ab06-5cc9fa7a25b1',
 'eab5ac97-f528-4c98-a92a-96d5fbea946d',
 'e31f2a67-be05-4ee2-91d6-3229c28bf917',
 '64e1e695-31f1-4542-bbbe-a5bf952300c9',
 'dde464e6-22b8-4a2f-beae-1eb4af637b55',
 '285b3bd6-aeb1-4aa1-8542-77c8a5c2b23a',
 'd60cbd1f-426a-4a40-acbe-2d935c1c2bbf',
 '2d7f7d09-f9d2-4906-8538-e11587249ecc',
 '2988eddb-97ad-4ed5-8b79-8b749bc94457',
 'a3953483-4ef1-41fb-a938-130bb219a7cf',
 '868fc5f1-73d2-4acc-8420-bb20d04b5d9e',
 'e03687e7-5906-483a-ace5-f631e0da5ac1',
 '3bd45922-8301-4373-905b-6207eb5c24a3',
 'cf869dd7-4167-4ec7-82c7-9b794365b104',
 '2c91e491-69ec-44fc-bd7f-db46ddd54c20',
 '311b55d5-d176-482e-a1d8-8d6f9250a02a',
 '548f00c7-f8a1-452f-971d-f0fd8ffb3e6b',
 '1f4e09d4-36e7-

In [6]:
txt_loader = TextLoader("Paestum/Paestum-Encyclopedia.txt")
txt_chunks = text_splitter.split_documents(txt_loader.load())
vector_db.add_documents(txt_chunks)

['e62165cf-1e1c-4b0c-9861-53d9dc80b288']

## Removing duplication

In [ ]:
def split_and_import(loader):
     chunks = text_splitter.split_documents(loader.load())
     vector_db.add_documents(chunks)
     print(f"Ingested chunks created by {loader}")

In [ ]:
try:
    wikipedia_loader = WikipediaLoader(query="Paestum")
    split_and_import(wikipedia_loader)
except Exception as error:
    print(f"Wikipedia API failed ({type(error).__name__}: {error}). Loading the Paestum page directly.")
    wikipedia_loader = WebBaseLoader(
        "https://en.wikipedia.org/wiki/Paestum",
        header_template={"User-Agent": "AI Agents and Agentic Workflows educational RAG notebook"}
    )
    split_and_import(wikipedia_loader)

word_loader = Docx2txtLoader("Paestum/Paestum-Britannica.docx")
split_and_import(word_loader)

pdf_loader = PyPDFLoader("Paestum/PaestumRevisited.pdf")
split_and_import(pdf_loader)

txt_loader = TextLoader("Paestum/Paestum-Encyclopedia.txt")
split_and_import(txt_loader)

## Ingesting Multiple Documents from a Folder (two techniques)

### 1) Iterating over all files in a folder

In [ ]:
loader_classes = {
    'docx': Docx2txtLoader,
    'pdf': PyPDFLoader,
    'txt': TextLoader
}

In [ ]:
import os

def get_loader(filename):
    _, file_extension = os.path.splitext(filename) #A Extract the file extension
    file_extension = file_extension.lstrip('.') #B Remove the leading dot from the extension

    loader_class = loader_classes.get(
        file_extension) #C Get the loader class from the dictionary

    if loader_class:
        return loader_class(filename) #D Instantiate and return the correct loader
    else:
        raise ValueError(f"No loader available for file extension '{file_extension}'")

### Ingesting the files from the folder

In [ ]:
folder_path = "CilentoTouristInfo" #A Path to the folder containing the documents

for filename in os.listdir(folder_path): #B iterate over the files in the path
    file_path = os.path.join(folder_path, filename) #C Construct the full path to the file

    if os.path.isfile(file_path): #D Check if it is a file (not a directory)
        try:
            loader = get_loader(file_path) #E Instantiate the correct loader for the file
            print(f"Loader for {filename}: {loader}")
            split_and_import(loader) #F Split and ingest
        except ValueError as e:
            print(e)

### 2) Ingesting all files with DirectoryLoader

In [ ]:
# Unstructured's local PDF inference extra does not support Windows Python 3.13.
# Use Unstructured locally for DOCX/TXT and the existing PyPDFLoader for PDF files.
# Requires: unstructured[docx]
# https://docs.langchain.com/oss/python/integrations/providers/unstructured
# https://docs.unstructured.io/open-source/installation/full-installation
folder_path = "CilentoTouristInfo"

unstructured_directory_loader = DirectoryLoader(
    folder_path, ["**/*.docx", "**/*.txt"]
) #A Load DOCX and TXT files with Unstructured
pdf_directory_loader = DirectoryLoader(
    folder_path, "**/*.pdf", loader_cls=PyPDFLoader
) #B Load PDFs locally with PyPDFLoader

split_and_import(unstructured_directory_loader)
split_and_import(pdf_directory_loader)

> **Windows/Python compatibility note:** Unstructured's local PDF inference dependency is not available for Windows Python 3.13. Therefore, this notebook uses Unstructured locally for DOCX/TXT files and the existing `PyPDFLoader` for PDFs. For full Unstructured PDF/OCR processing, use Python 3.12, install the PDF extra, and provide the required system tools such as Poppler and Tesseract. See the [LangChain Unstructured integration](https://docs.langchain.com/oss/python/integrations/providers/unstructured) and [Unstructured full-installation guide](https://docs.unstructured.io/open-source/installation/full-installation).
>
> Repeated `libmagic is unavailable` messages are non-fatal file-type-detection advisories. This notebook supplies explicit `.docx`, `.txt`, and `.pdf` glob patterns, so files with correct extensions can still be loaded without native `libmagic`. The two `Ingested chunks created by ... DirectoryLoader` messages confirm that both loader paths—Unstructured for DOCX/TXT and `PyPDFLoader` for PDF—completed successfully. Native `libmagic` is mainly useful here for files with missing, incorrect, or ambiguous extensions.

## Querying the vector store directly

In [7]:
query = "Where was Poseidonia and who renamed it to Paestum?"
results = vector_db.similarity_search(query, 4) # four clostest results
print(results)

[Document(id='5afc7d6f-072c-4495-a3b1-a2d850dddcb8', metadata={'source': 'https://en.wikipedia.org/wiki/Paestum', 'title': 'Paestum - Wikipedia', 'language': 'en'}, page_content='The Greek settlers who founded the city originally named it Poseidonia (Ancient Greek: Ποσειδωνία). It was eventually conquered by the local Lucanians and later the Romans. The Lucanians renamed it to Paistos and the Romans gave the city its current name.[5]\nAncient ruins and features[edit]\nAerial view of Paestum, looking north; two Hera Temples in foreground, Athena Temple in background.'), Document(id='419e5545-b92a-450b-adab-90cdb7e1611f', metadata={'source': 'https://en.wikipedia.org/wiki/Paestum', 'title': 'Paestum - Wikipedia', 'language': 'en'}, page_content="located there, after which the city would have been named. The date of Poseidonia's founding is not given by ancient sources, but the archaeological evidence gives a date of approximately 600\xa0BCE.[19]"), Document(id='818be703-fa15-41fa-81e8-75

In [8]:
len(results)

4

## Asking a question through a LangChain's RAG chain

In [9]:
from langchain_core.prompts import PromptTemplate

rag_prompt_template = """Use the following pieces of context
to answer the question at the end.
If you don't know the answer, just say that you don't know,
don't try to make up an answer.
Use three sentences maximum and keep the
answer as concise as possible.
{context}
Question: {question}
Helpful Answer:"""

rag_prompt = PromptTemplate.from_template(rag_prompt_template)

Alternatively, you can pull the prompt instance directly from the **[LangChain Hub](https://smith.langchain.com/hub)**:

```Python
from langchain import hub
rag_prompt = hub.pull("rlm/rag-prompt")
```

In [10]:
retriever = vector_db.as_retriever()

In [11]:
from langchain_core.runnables import RunnablePassthrough
question_feeder = RunnablePassthrough()

`RunnablePassthrough` is a core component in LangChain's LangChain Reference Expression Language (LCEL) that _takes an input and **returns it completely unchanged**_. It acts like an identity function, making it essential for passing raw data—like a user's original question—alongside processed intermediate data in complex multi-step chains.
- **Retrieval-Augmented Generation (RAG) Use Case**: Pass the original user query straight through to a prompt template while a separate retriever branch fetches context.

In [12]:
chatbot = ChatOllama(
    model="gemma4:12b-it-q8_0",
    num_ctx=65536,
    temperature=0,
    reasoning=False
)

In [13]:
# set up RAG chain

rag_chain = {"context": retriever,
             "question": question_feeder} | rag_prompt | chatbot

Each block in a LangChain chain implements the `Runnable interface` and **accepts a dictionary as input**. This is why the first block in the chain is a dictionary.

In [14]:
def execute_chain(chain, question):
    answer = chain.invoke(question)
    return answer

In [15]:
question = """Where was Poseidonia and who renamed
it to Paestum. Also tell me the source."""
answer = execute_chain(rag_chain, question)
print(answer.content)

Poseidonia was an ancient Greek colony located in southern Italy, approximately 22 miles southeast of modern Salerno. The Romans gave the city its current name of Paestum after it was previously renamed Paistos by the Lucanians. This information is sourced from Wikipedia and Britannica.


In [16]:
print(answer)

content='Poseidonia was an ancient Greek colony located in southern Italy, approximately 22 miles southeast of modern Salerno. The Romans gave the city its current name of Paestum after it was previously renamed Paistos by the Lucanians. This information is sourced from Wikipedia and Britannica.' additional_kwargs={} response_metadata={'model': 'gemma4:12b-it-q8_0', 'created_at': '2026-08-09T06:13:52.2597852Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2364077500, 'load_duration': 213704100, 'prompt_eval_count': 669, 'prompt_eval_duration': 459076000, 'eval_count': 57, 'eval_duration': 1486659000, 'logprobs': None, 'model_name': 'gemma4:12b-it-q8_0', 'model_provider': 'ollama'} id='lc_run--019fe527-f8c6-7fc0-ae83-3172b42dc2fc-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 669, 'output_tokens': 57, 'total_tokens': 726}


In [17]:
# Follow-up question to check if there is any memory of the previous question and answer.

question = """And then, what they do?
Tell me only if you know.
Also tell me the source"""
answer = execute_chain(rag_chain, question)
print(answer.content)

I do not know what "they" do because the provided text does not contain information regarding actions or activities performed by any specific subjects. The documents consist of lists of place names and metadata related to Paestum.


It is obvious that the ***chatbot has no memory of previous dialog context*** and doesn’t understand that “they” refers to the Romans. Currently, it’s **stateless** and simply passes questions from the user to the LLM and back, without retaining any memory of the conversation flow. Let’s now add the memory to the chatbot.

## Chatbot memory of message history

In [18]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables import RunnableLambda

rag_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant, world-class expert in Roman and Greek history, especially in towns located in southern Italy. Provide interesting insights on local history and recommend places to visit with knowledgeable and engaging answers. Answer all questions to the best of your ability, but only use what has been provided in the context. If you don't know, just say you don't know. Use three sentences maximum and keep the answer as concise as possible."),
        ("placeholder", "{chat_history_messages}"),
        ("assistant", "{retrieved_context}"),
        ("human", "{question}"),
    ]
)

retriever = vector_db.as_retriever()
question_feeder = RunnablePassthrough()
chatbot = ChatOllama(
    model="gemma4:12b-it-q8_0",
    num_ctx=65536,
    temperature=0,
    reasoning=False
)
chat_history_memory = ChatMessageHistory()

def get_messages(x):
    return chat_history_memory.messages

rag_chain = {
    "retrieved_context": retriever,
    "question": question_feeder,
    "chat_history_messages": RunnableLambda(get_messages)
} | rag_prompt | chatbot

def execute_chain_with_memory(chain, question):
    chat_history_memory.add_user_message(question)
    answer = chain.invoke(question)
    chat_history_memory.add_ai_message(answer)
    print(f'Full chat message history: {chat_history_memory.messages}\n\n')
    return answer

`RunnablePassthrough` takes an input and returns it completely unchanged, whereas `RunnableLambda` wraps a custom function or lambda expression to process, alter, or transform the input data inside a LangChain Expression Language (LCEL) pipeline.

**Core Differences**
- `RunnablePassthrough`: Acts as an identity function (`f(x) = x`). It forwards data untouched, typically used in parallel blocks to carry original inputs forward alongside transformed data or to assign new keys via .assign().
- `RunnableLambda`: Executes custom user logic. It converts any regular Python or JavaScript function into a valid pipeline component to change the data format, parse text, or call external APIs.

**Key Features**

***RunnablePassthrough***
- **Passes raw data**: Outputs whatever input it receives without executing code.
- **Retains context**: Keeps the initial user prompt or variable available for later steps in a chain.
- **Supports assignment**: Includes an `.assign()` method to add new calculated fields into an existing input dictionary.

***RunnableLambda***
- **Transforms data**: Applies custom code, filters, or calculations to the incoming value.
- **Wraps functions**: Turns standard functions into unified components that support `.invoke()`, `.batch()`, and async calls.
- **Handles logic**: Ideal for data validation, formatting strings, or routing decisions inside a sequence.

In [19]:
question = """Where was Poseidonia and who renamed
it to Paestum? Also tell me the source."""
answer = execute_chain_with_memory(rag_chain, question)
print(answer.content)

Full chat message history: [HumanMessage(content='Where was Poseidonia and who renamed\nit to Paestum? Also tell me the source.', additional_kwargs={}, response_metadata={}), AIMessage(content='Poseidonia was an ancient Greek colony located in southern Italy, approximately 22 miles southeast of modern Salerno. The city was renamed Paistos by the local Lucanians, while the Romans eventually gave it its current name. This information is sourced from Wikipedia and Britannica.', additional_kwargs={}, response_metadata={'model': 'gemma4:12b-it-q8_0', 'created_at': '2026-08-09T06:13:56.4325476Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2046308500, 'load_duration': 210269500, 'prompt_eval_count': 726, 'prompt_eval_duration': 290665000, 'eval_count': 55, 'eval_duration': 1337778000, 'logprobs': None, 'model_name': 'gemma4:12b-it-q8_0', 'model_provider': 'ollama'}, id='lc_run--019fe528-0a51-7d03-bd72-d1cc33010126-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_to

In [20]:
# Follow-up question to check if there is any memory of the previous question and answer.

question = """And then what did they do?
Also tell me the source"""
answer = execute_chain_with_memory(rag_chain, question)
print(answer.content)

Full chat message history: [HumanMessage(content='Where was Poseidonia and who renamed\nit to Paestum? Also tell me the source.', additional_kwargs={}, response_metadata={}), AIMessage(content='Poseidonia was an ancient Greek colony located in southern Italy, approximately 22 miles southeast of modern Salerno. The city was renamed Paistos by the local Lucanians, while the Romans eventually gave it its current name. This information is sourced from Wikipedia and Britannica.', additional_kwargs={}, response_metadata={'model': 'gemma4:12b-it-q8_0', 'created_at': '2026-08-09T06:13:56.4325476Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2046308500, 'load_duration': 210269500, 'prompt_eval_count': 726, 'prompt_eval_duration': 290665000, 'eval_count': 55, 'eval_duration': 1337778000, 'logprobs': None, 'model_name': 'gemma4:12b-it-q8_0', 'model_provider': 'ollama'}, id='lc_run--019fe528-0a51-7d03-bd72-d1cc33010126-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_to

## Verifying conversational retrieval separately from memory

The saved follow-up answer confirms that **message memory works for answer generation**: the model understands that "they" refers to the people discussed in the preceding exchange and that the question concerns events after the renaming. Its refusal is appropriate grounded behavior because the retrieved context does not provide the requested later actions. The [current book notebook](https://github.com/roberto-inf/building-llm-applications/blob/main/ch07/07-QA_across_documents.ipynb) now records the same conservative result with `gpt-5-nano`, so this outcome is not evidence that the local model lacks relevant internal knowledge.

However, this chain is **not history-aware at the retrieval stage**. LangChain retrievers accept a string query and return matching documents; here, `retriever` receives only the literal current question, such as "And then what did they do?", while `chat_history_messages` is supplied separately to the final model prompt. Consequently, the model can resolve the pronoun from memory, but Chroma searches using an ambiguous query that does not mention the Romans, Poseidonia, Paestum, or the renaming. See the [LangChain retriever documentation](https://docs.langchain.com/oss/python/integrations/retrievers).

A production conversational RAG pipeline should first **rewrite the follow-up as a standalone search query** using the relevant message history, retrieve with that explicit query, and only then generate an answer grounded in the retrieved documents. It should also manage per-session history with a dedicated history wrapper such as `RunnableWithMessageHistory`. The current helper is suitable for illustrating the concept, but it adds the current user message before invoking the chain, so that question appears both in the history placeholder and again in the final human-message slot.

The following diagnostics isolate retrieval from generation. They compare exactly what the current chain retrieves for the ambiguous follow-up with what the same retriever returns for an equivalent standalone query. Run both cells after creating and populating `vector_db`.

In [21]:
# Inspect the documents retrieved from the literal, ambiguous follow-up.
follow_up_question = "And then what did they do? Also tell me the source"
literal_follow_up_docs = retriever.invoke(follow_up_question)

def print_retrieved_documents(label, documents):
    print(f"{label}: {len(documents)} document(s)\n")
    for number, document in enumerate(documents, start=1):
        source = document.metadata.get("source", "Unknown source")
        print(f"--- Retrieved document {number} ---")
        print(f"Source: {source}")
        print(document.page_content[:1000])
        print()

print_retrieved_documents(
    "Literal follow-up retrieval", literal_follow_up_docs
)

Literal follow-up retrieval: 4 document(s)

--- Retrieved document 1 ---
Source: https://en.wikipedia.org/wiki/Paestum
Retrieved from "https://en.wikipedia.org/w/index.php?title=Paestum&oldid=1364944865"

--- Retrieved document 2 ---
Source: https://en.wikipedia.org/wiki/Paestum
were 24 square or round towers. There may have been as many as 28, but some of them (and Porta Aurea) were destroyed during the construction of a highway during the 18th century that effectively cut the ancient site in two.

--- Retrieved document 3 ---
Source: https://en.wikipedia.org/wiki/Paestum
Second World War[edit]
A US Army company with its transceiver office between the Doric columns of the Temple of Neptune, 22 Sept 1943

--- Retrieved document 4 ---
Source: https://en.wikipedia.org/wiki/Paestum
3.4
Roman period and abandonment










4
Rediscovery




Toggle Rediscovery subsection





4.1
Second World War








4.2
Recent developments










5
Coins








6
In fiction








7
See also





In [22]:
# Simulate the query that a history-aware rewriting stage should produce.
standalone_follow_up_query = (
    "What did the Romans do after renaming Poseidonia to Paestum? "
    "Also identify the source."
)
standalone_follow_up_docs = retriever.invoke(standalone_follow_up_query)

print(f"Standalone query: {standalone_follow_up_query}\n")
print_retrieved_documents(
    "Standalone-query retrieval", standalone_follow_up_docs
)

Standalone query: What did the Romans do after renaming Poseidonia to Paestum? Also identify the source.

Standalone-query retrieval: 4 document(s)

--- Retrieved document 1 ---
Source: https://en.wikipedia.org/wiki/Paestum
The Greek settlers who founded the city originally named it Poseidonia (Ancient Greek: Ποσειδωνία). It was eventually conquered by the local Lucanians and later the Romans. The Lucanians renamed it to Paistos and the Romans gave the city its current name.[5]
Ancient ruins and features[edit]
Aerial view of Paestum, looking north; two Hera Temples in foreground, Athena Temple in background.

--- Retrieved document 2 ---
Source: https://en.wikipedia.org/wiki/Paestum
Higginbotham, James (2012). "Paestum (Poseidonia)". The Encyclopedia of Ancient History. Blackwell Publishing Ltd. doi:10.1002/9781444338386.wbeah16104. ISBN 978-1-4443-3838-6.
Horsnaes, Helle W. (2002). The Cultural Development in North Western Lucania c. 600–273 BCE. Analecta Romana Instituti Danici Suppl

### How to interpret the evidence

The executed diagnostics provide decisive evidence about this pipeline:

- **Conversation memory works for answer generation.** In the follow-up response, Gemma resolves the ambiguous pronoun "they" as the Lucanians or Romans from the preceding exchange.
- **Retrieval is not history-aware.** Searching with the literal follow-up retrieved unrelated fragments from `Parmenides.docx`, the Wikipedia article about red-figure pottery, and `Velia.pdf`. The retriever received only the current ambiguous question; it did not receive or rewrite it using the chat history.
- **The indexed corpus contains a relevant partial answer.** Searching with the standalone query retrieved Paestum passages stating that the Romans took over in 273 BCE, renamed the city Paestum, and established a Latin colony.
- **Gemma's refusal was appropriately grounded.** That relevant passage was not among the documents retrieved for the literal follow-up, so answering from internal model knowledge would have violated the instruction to use only the provided context. The result therefore does not demonstrate weaker internal knowledge than the OpenAI model; it demonstrates a retrieval-query limitation.

The recorded run was also not a clean first-ingestion-only test. Its execution history shows that the original Paestum ingestion cells, the alternative Paestum ingestion block, and the `CilentoTouristInfo` ingestion loop were executed. This explains the unrelated Cilento results and repeated identical Paestum chunks. For a clean comparison, restart the kernel to clear the in-memory Chroma collection and message history, run only one ingestion approach, and then rerun the RAG, memory, and diagnostic cells.

**Final conclusion:** the current chain successfully demonstrates generation-side conversational memory, but it is not yet a complete conversational RAG implementation. A production-quality pipeline should use the relevant message history to rewrite each follow-up into a standalone query before retrieval, then generate an answer grounded only in the documents retrieved for that rewritten query.

## Production-quality conversational RAG

This implementation separates a conversational RAG turn into two grounded stages:

1. **Query contextualization:** the local model uses the previous messages only to rewrite the latest user message as a self-contained search query. It does not answer the question at this stage.
2. **Retrieval and grounded generation:** Chroma retrieves documents using that standalone query. The model then receives the current retrieved context and the conversation history, but the prompt permits history only for understanding conversational references; every factual claim must be supported by the documents retrieved for the current turn.

LangGraph's checkpointed state maintains a separate message history for each `thread_id`, avoiding the manual pre-invocation update that duplicated the current question in the earlier educational chain. This local notebook uses `InMemorySaver`; a deployed service should replace it with a durable checkpointer appropriate to its database and concurrency requirements. The returned state exposes `standalone_query`, `source_documents`, and `answer` so the complete retrieval path can be inspected.

> **Clean-run note:** The saved execution below was produced in a fresh kernel with only the first Wikipedia/DOCX/PDF/TXT ingestion approach. The alternative Paestum ingestion block and both folder-ingestion demonstrations were deliberately left unexecuted. Therefore, the earlier interpretation cell's note about a previously mixed ingestion run documents the earlier evidence, not this final saved execution.

In [23]:
import warnings

from langchain_core._api.deprecation import LangChainPendingDeprecationWarning
from langchain_core.documents import Document
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser

# Suppress only a dependency-internal pending warning from the pinned versions.
with warnings.catch_warnings():
    warnings.filterwarnings(
        "ignore",
        message=r"The default value of `allowed_objects` will change.*",
        category=LangChainPendingDeprecationWarning,
    )
    from langgraph.checkpoint.memory import InMemorySaver
    from langgraph.graph import END, START, MessagesState, StateGraph

production_chatbot = ChatOllama(
    model="gemma4:12b-it-q8_0",
    num_ctx=65536,
    temperature=0,
    reasoning=False
)
production_retriever = vector_db.as_retriever(search_kwargs={"k": 4})

contextualize_question_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Rewrite the latest user question as a self-contained search query. "
            "Resolve pronouns and omitted references using the chat history. "
            "Resolve a pronoun to the actor of the most recent relevant action; "
            "do not merge distinct actors unless the user explicitly refers to both. "
            "Do not answer the question and do not add facts. If the question is "
            "already self-contained, return it unchanged. For example, if the history "
            "says that the Lucanians renamed the city Paistos and the Romans later gave "
            "it the name Paestum, rewrite 'And then what did they do?' as 'What did the "
            "Romans do after renaming Poseidonia to Paestum?' Output only the query."
        ),
        ("placeholder", "{chat_history}"),
        ("human", "{input}"),
    ]
)
question_rewriter = (
    contextualize_question_prompt | production_chatbot | StrOutputParser()
)

def format_documents_for_prompt(documents):
    return "\n\n".join(
        f"[Source: {document.metadata.get('source', 'Unknown source')}]\n"
        f"{document.page_content}"
        for document in documents
    )

class ProductionRAGState(MessagesState):
    standalone_query: str
    source_documents: list[Document]
    context: str
    answer: AIMessage

def rewrite_and_retrieve(state):
    latest_question = state["messages"][-1].content
    chat_history = state["messages"][:-1]
    if chat_history:
        standalone_query = question_rewriter.invoke(
            {"input": latest_question, "chat_history": chat_history}
        ).strip()
    else:
        standalone_query = latest_question

    source_documents = production_retriever.invoke(standalone_query)
    return {
        "standalone_query": standalone_query,
        "source_documents": source_documents,
        "context": format_documents_for_prompt(source_documents),
    }

grounded_answer_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Answer the user's question using only the retrieved context below. "
            "Use chat history only to understand conversational references; never "
            "treat it as factual evidence. Every factual claim must be supported by "
            "the current retrieved context. If the context is insufficient, say you "
            "do not know. Use no more than three concise sentences and cite the source "
            "labels supplied in the context.\n\nRetrieved context:\n{context}"
        ),
        ("placeholder", "{chat_history}"),
        ("human", "{input}"),
    ]
)

grounded_answer_chain = grounded_answer_prompt | production_chatbot

def generate_grounded_answer(state):
    latest_question = state["messages"][-1].content
    chat_history = state["messages"][:-1]
    answer = grounded_answer_chain.invoke(
        {
            "input": latest_question,
            "chat_history": chat_history,
            "context": state["context"],
        }
    )
    return {"messages": [answer], "answer": answer}

production_workflow = StateGraph(ProductionRAGState)
production_workflow.add_node("rewrite_and_retrieve", rewrite_and_retrieve)
production_workflow.add_node("generate_grounded_answer", generate_grounded_answer)
production_workflow.add_edge(START, "rewrite_and_retrieve")
production_workflow.add_edge("rewrite_and_retrieve", "generate_grounded_answer")
production_workflow.add_edge("generate_grounded_answer", END)

production_checkpointer = InMemorySaver()
production_rag_graph = production_workflow.compile(
    checkpointer=production_checkpointer
)
production_config = {"configurable": {"thread_id": "paestum-demo"}}

In [24]:
# First turn: no rewriting is necessary because the question is self-contained.
production_first_question = (
    "Where was Poseidonia and who renamed it to Paestum? "
    "Also tell me the source."
)
production_first_result = production_rag_graph.invoke(
    {"messages": [HumanMessage(content=production_first_question)]},
    config=production_config,
)

print(f"Retrieval query: {production_first_result['standalone_query']}\n")
print(f"Answer: {production_first_result['answer'].content}\n")
print("Retrieved sources:")
for document in production_first_result["source_documents"]:
    print(f"- {document.metadata.get('source', 'Unknown source')}")

Retrieval query: Where was Poseidonia and who renamed it to Paestum? Also tell me the source.

Answer: Poseidonia was an ancient Greek colony located in southern Italy near the west coast, approximately 22 miles southeast of modern Salerno [Paestum/Paestum-Britannica.docx]. The city was renamed to Paistos by the local Lucanians and eventually received its current name from the Romans [https://en.wikipedia.org/wiki/Paestum].

Retrieved sources:
- https://en.wikipedia.org/wiki/Paestum
- https://en.wikipedia.org/wiki/Paestum
- Paestum/Paestum-Britannica.docx
- https://en.wikipedia.org/wiki/Paestum


In [25]:
# Follow-up turn: history is used to rewrite the ambiguous question before retrieval.
production_follow_up = "And then what did they do? Also tell me the source."
production_follow_up_result = production_rag_graph.invoke(
    {"messages": [HumanMessage(content=production_follow_up)]},
    config=production_config,
)

print(f"Original follow-up: {production_follow_up}")
print(f"Rewritten retrieval query: {production_follow_up_result['standalone_query']}\n")
print(f"Grounded answer: {production_follow_up_result['answer'].content}\n")
print_retrieved_documents(
    "Documents retrieved with the rewritten query",
    production_follow_up_result["source_documents"],
)
print("Production chat history:")
production_state = production_rag_graph.get_state(production_config)
for message in production_state.values["messages"]:
    print(f"{message.type}: {message.content}")

Original follow-up: And then what did they do? Also tell me the source.
Rewritten retrieval query: What did the Romans do after renaming Poseidonia to Paestum, and what is the source?

Grounded answer: After taking over in 273 BCE, the Romans renamed the city Paestum and established a Latin colony [https://en.wikipedia.org/wiki/Paestum]. The city later declined due to shifts in trade routes and the onset of flooding and marsh formation [https://en.wikipedia.org/wiki/Paestum].

Documents retrieved with the rewritten query: 4 document(s)

--- Retrieved document 1 ---
Source: https://en.wikipedia.org/wiki/Paestum
The Greek settlers who founded the city originally named it Poseidonia (Ancient Greek: Ποσειδωνία). It was eventually conquered by the local Lucanians and later the Romans. The Lucanians renamed it to Paistos and the Romans gave the city its current name.[5]
Ancient ruins and features[edit]
Aerial view of Paestum, looking north; two Hera Temples in foreground, Athena Temple in ba

## Tracing with LangSmith

Stop the notebook and open a new operative system shell (for example Windows command shell).

Configure the relevant environment variables in the OS shell, the rerun the previous Jupyter cells:
```
(env_ch07) C:\...\ch07>set LANGSMITH_TRACING=true
(env_ch07) C:\...\ch07>set LANGSMITH_ENDPOINT=https://api.smith.langchain.com
(env_ch07) C:\...\ch07>set LANGSMITH_PROJECT=Q & A chatbot
(env_ch07) C:\...\ch07>set LANGSMITH_API_KEY=<YOUR_LANGSMITH_API_KEY>
```
Then Restart the Jupyter notebook:

```(env_ch07) C:\...\ch07>jupyter notebook 07-QA_across_documents.ipynb```

Finally re-execute the whole Jupyter notebook cell by cell. All the activity will have not been logged through LangSmith.